# Global Bollinger EODHD v4\n\nUpload the current source ZIP, keep your Colab `EODHD_API_TOKEN` in the notebook Secrets panel, run validation tests, then build `netlify_site.zip`. This notebook does not publish to Netlify by itself.

In [ ]:
import os\nimport sys\nimport json\nimport zipfile\nimport subprocess\nfrom pathlib import Path\n\nfrom google.colab import files, userdata\nfrom IPython.display import display\n\nROOT = Path('/content/mk_bb_eodhd')\nROOT.mkdir(parents=True, exist_ok=True)\nBUILD_OK = False\n\ntry:\n    token = userdata.get('EODHD_API_TOKEN')\nexcept Exception:\n    raise RuntimeError(\n        'Add EODHD_API_TOKEN in Colab Secrets and enable notebook access. '\n        'GitHub Secrets are not shared with Colab.'\n    ) from None\n\nif not token or not token.strip():\n    raise RuntimeError('EODHD_API_TOKEN is empty or unavailable.')\n\nos.environ['EODHD_API_TOKEN'] = token.strip()\ndel token\n\nprint('Select the current MK_BB_EODHD_Colab_Netlify.zip source package.')\nuploaded = files.upload()\narchives = [name for name in uploaded if name.lower().endswith('.zip')]\n\nif len(archives) != 1:\n    raise RuntimeError('Please upload exactly one source ZIP file.')\n\narchive_path = Path(archives[0]).resolve()\nrequired_files = {\n    'bb_eodhd.py', 'analytics.py', 'commodities.py', 'netlify_setup.py',\n    'report_ui.py', 'portal.html', 'portal.css', 'portal.js',\n    'requirements.txt', 'test_engine.py', 'test_analytics.py',\n    'config.json', 'universe.json'\n}\n\nwith zipfile.ZipFile(archive_path) as archive:\n    names = set(archive.namelist())\n    missing = required_files - names\n    if missing:\n        raise RuntimeError('This is not the current v4 source package. Missing files: ' + ', '.join(sorted(missing)))\n\n    for member in archive.infolist():\n        destination = (ROOT / member.filename).resolve()\n        if not destination.is_relative_to(ROOT.resolve()):\n            raise RuntimeError('Unsafe file path inside the ZIP.')\n\n    for member in archive.infolist():\n        destination = ROOT / member.filename\n        if member.filename in {'config.json', 'universe.json'} and destination.exists():\n            print('Existing settings preserved:', member.filename)\n            continue\n        archive.extract(member, ROOT)\n\nos.chdir(ROOT)\n\ndef run_step(title, arguments):\n    print('\\n' + title, flush=True)\n    subprocess.run([sys.executable, *arguments], cwd=ROOT, check=True)\n

In [ ]:
run_step('1/4 - Installing Python dependencies', ['-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])\nrun_step('2/4 - Running validation tests', ['-m', 'unittest', 'discover', '-v'])\nrun_step('3/4 - Checking configuration', ['bb_eodhd.py', '--init'])\n

## Build the Netlify site ZIP\n\nThis step fetches real EODHD daily data and builds `netlify_site.zip`. Missing, unsupported or non-daily instruments appear in the portal's Data Audit tab; they are not replaced by fallback, resampling or synthetic data.

In [ ]:
try:\n    run_step('4/4 - Fetching EODHD data and building the report', ['bb_eodhd.py'])\nexcept subprocess.CalledProcessError:\n    audit_path = ROOT / 'private' / 'latest_audit.json'\n    if audit_path.exists():\n        import pandas as pd\n        print('\\nLast saved Data Audit:')\n        display(pd.DataFrame(json.loads(audit_path.read_text(encoding='utf-8'))))\n    raise RuntimeError(\n        'This run did not complete. Share the error output above. '\n        'The previously generated ZIP has not been replaced.'\n    ) from None\n\nsite_zip = ROOT / 'netlify_site.zip'\nif not site_zip.exists():\n    raise RuntimeError('netlify_site.zip not found after the run.')\n\nwith zipfile.ZipFile(site_zip) as archive:\n    if 'index.html' not in archive.namelist():\n        raise RuntimeError('index.html missing from the site package.')\n    if 'portal-data.js' not in archive.namelist():\n        raise RuntimeError('portal-data.js missing from the site package.')\n    failed = archive.testzip()\n    if failed is not None:\n        raise RuntimeError('Site ZIP integrity check failed at: ' + failed)\n\nBUILD_OK = True\nprint('\\nSUCCESS: netlify_site.zip with index.html created.')\nprint('Review Data Audit before publishing. Not all registered assets are guaranteed to be available from EODHD.')\nfiles.download(str(site_zip))\n

## Optional GitHub Actions deployment\n\nFor daily automation, upload this source package to the private GitHub repository root. Configure `EODHD_API_TOKEN`, `NETLIFY_AUTH_TOKEN` and `NETLIFY_SITE_ID` as Actions Secrets. Set the repository variable `PUBLISH_APPROVED=true` only after the first manual review of access, licensing and display quality.